# SL ladder · 02 · The linked analysis table

Joins outcome, population and exposure into one district-week table with exactly the columns the
frozen ladder code reads. Getting this schema right is what lets `sl_04` run the frozen design
**verbatim** instead of paraphrasing it.

Two conventions in here were *not* documented anywhere and had to be recovered empirically against the
frozen labels. They are derived in `sl_03`; this notebook applies the answers and states them plainly.

## 1 · Setup

In [1]:
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd().parent if Path.cwd().name.startswith("notebooks") else Path.cwd()
DQ   = REPO / "data_quarantine"
ERA  = DQ / "wp5_exposure" / "era5_0p25_window"
CHI  = DQ / "wp5_exposure" / "chirps_window"
OUTD = DQ / "sl_ladder"
OUTD.mkdir(parents=True, exist_ok=True)

o = pd.read_csv(DQ / "wer_srilanka/frozen/wer_dengue_currentweek_rdhs_2018_2025_v2.1-refresh.csv")
o = o.rename(columns={"year": "epi_year", "week": "epi_week"})
ex = pd.read_csv(OUTD / "sl_exposure_weekly_0p25_area.csv", parse_dates=["week_start"])
pop = pd.read_csv(DQ / "m6_geomatics/m6_batchB_population_by_year.csv")
we = pd.read_csv(DQ / "wp5_exposure/wp5_buildB_weights_era5_0p25_srilanka_2020.csv")
xw = we[["geometry_id", "rdhs_name"]].drop_duplicates()
print(o.shape, ex.shape, pop.shape)

(10790, 9) (10842, 5) (78, 5)


## 2 · Reporting week to calendar date

**Convention 1.** The WER numbers a reporting week, and the frozen table keys rows by a Monday. The
obvious mapping - ISO Monday of week *w* - is **wrong**: it scores 0.820 agreement against the 3,926
frozen labels, where shifting back one week scores 0.988.

The reading that fits: the bulletin labelled week *w* carries the week that has just ended. So
`week_start = ISO_Monday(y, w) - 7d`. `sl_03` shows the full sweep this came from.

In [2]:
def iso_monday(y, w):
    try:
        return pd.Timestamp(pd.Timestamp.fromisocalendar(int(y), int(w), 1))
    except ValueError:
        return pd.NaT   # a year with no ISO week 53

wk = o[["epi_year", "epi_week"]].drop_duplicates()
wk["week_start"] = [iso_monday(y, w) - pd.Timedelta(days=7) for y, w in zip(wk.epi_year, wk.epi_week)]
print("reporting weeks with no ISO date:", int(wk.week_start.isna().sum()))
o = o.merge(wk, on=["epi_year", "epi_week"], how="left")
o = o[o.week_start.notna()].copy()
o = o.merge(xw, left_on="rdhs", right_on="rdhs_name", how="left")
assert o.geometry_id.notna().all(), "unmatched district name"

reporting weeks with no ISO date: 1


## 3 · Population denominator

**Convention 2.** A year-varying denominator scores 0.968 label agreement; a **time-invariant** one
scores 0.983. The frozen run behaved as though population were constant per district, so that is what
is used here.

This is convenient as well as faithful: with a constant denominator the label - a within-district
quantile of incidence - is scale-invariant, so the 2021-2025 WorldPop years missing from this machine
stop mattering at all.

In [3]:
pop = pop.merge(xw, on="rdhs_name", how="left")
pop2020 = pop[pop.year == 2020].set_index("geometry_id")["pop_sum"]
popf = pd.DataFrame([{"geometry_id": g, "epi_year": y, "population": float(pop2020[g])}
                     for g in pop2020.index for y in range(2018, 2026)])
o = o.merge(popf, on=["geometry_id", "epi_year"], how="left")
o["dengue_incidence_per_100k"] = o["dengue_current_week"] / o["population"] * 1e5
print(o[["dengue_current_week", "population", "dengue_incidence_per_100k"]].describe().round(2).to_string())

       dengue_current_week  population  dengue_incidence_per_100k
count             10705.00    10764.00                   10705.00
mean                 37.98   812338.03                       4.73
std                  71.91   618666.18                      15.79
min                   0.00    53643.46                       0.00
25%                   4.00   424812.76                       0.95
50%                  14.00   680545.83                       2.40
75%                  40.00  1076313.82                       5.05
max                 990.00  2437061.73                    1046.25


## 4 · Join, and flag rather than impute

Missingness is carried as flags, exactly as the frozen table did - the ladder code filters on them.
Nothing is filled in here.

In [4]:
m = o.merge(ex, on=["geometry_id", "week_start"], how="left")
m["week_end"] = m["week_start"] + pd.Timedelta(days=6)
m["outcome_missing_flag"]    = m["dengue_current_week"].isna().astype(int)
m["exposure_missing_flag"]   = m[["t2m_mean_c", "precip_sum_mm", "rh_mean_percent"]].isna().any(axis=1).astype(int)
m["population_missing_flag"] = m["population"].isna().astype(int)

cols = ["geometry_id", "rdhs_name", "week_start", "week_end", "epi_year", "epi_week",
        "dengue_current_week", "population", "dengue_incidence_per_100k",
        "t2m_mean_c", "precip_sum_mm", "rh_mean_percent",
        "outcome_missing_flag", "exposure_missing_flag", "population_missing_flag"]
m = m[cols].sort_values(["geometry_id", "week_start"]).reset_index(drop=True)
print("linked", m.shape)
print({c: int(m[c].sum()) for c in cols[-3:]})
print("clean rows:", int((m[cols[-3:]].sum(axis=1) == 0).sum()))

linked (10764, 15)
{'outcome_missing_flag': 59, 'exposure_missing_flag': 26, 'population_missing_flag': 0}
clean rows: 10679


## 5 · Does every frozen row exist here?

The strongest available check on conventions 1 and 2 together: every one of the 3,926 frozen
`(district, predictor_week)` keys must be present. A week-numbering error would show up here as
missing keys.

In [5]:
fz = pd.read_csv(REPO / "ALT_STATS/frozen/srilanka_matched_pairs.csv", parse_dates=["predictor_week"])
fz = fz[fz.setting == "SriLanka"]
have = set(map(tuple, m[["geometry_id", "week_start"]].astype({"week_start": str}).values))
want = set(map(tuple, fz[["spatial_unit_id", "predictor_week"]].astype({"predictor_week": str}).values))
print(f"frozen keys present in rebuilt table: {len(want & have)}/{len(want)}")
assert want <= have, "rebuilt table is missing frozen keys"

m.to_csv(OUTD / "sl_linked_v2equiv.csv", index=False)
print("wrote", (OUTD / "sl_linked_v2equiv.csv").relative_to(REPO))

frozen keys present in rebuilt table: 3926/3926
wrote data_quarantine/sl_ladder/sl_linked_v2equiv.csv
